# Mission 3 TF-IDF + Logistic Regression baseline

이 노트북은 저장소의 현재 `scripts/mission3_baseline.py`와 `scripts/inference.py`를 기준으로 실행합니다.

## 공식 규정 확인 기준

- K-ICT 빅데이터센터 공식 메일 **`[2026 DCC] 대학부 규정 안내` (2026-09-21)**: 미션별 `inference.py`를 한 번 실행해 결과 파일을 생성할 수 있어야 하며, `requirements.txt`로 실행 환경을 재현할 수 있어야 합니다.
- 같은 공식 메일의 Decision Threshold 규정: **모든 class에 동일하게 0.5를 고정**하며, Train/Validation/OOF를 이용한 threshold tuning도 허용되지 않습니다.
- Mission 3 출제 규칙에 따라 모델 입력은 **대화 전사 텍스트**만 사용합니다. `startAt`, `id`, `recordId` 등 다른 annotation은 모델 입력·정렬·분할에 사용하지 않습니다.
- 학습 정답은 Training JSON의 `symptom` 중 평가 대상 9개만 사용합니다.
- 내부 검증은 Training을 80:20 multilabel stratified split, seed=42로 나눕니다.
- 공식 Validation은 학습·threshold 조정에 사용하지 않고 최종 추론 확인에만 사용합니다.


In [ ]:
# 먼저 이 비공개 저장소를 Colab 또는 로컬에 준비한 뒤 실행합니다.
# 기존 파일을 자동으로 덮어쓰거나 다른 브랜치로 바꾸지 않습니다.
from pathlib import Path
import os
import sys
import subprocess

current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (current, *current.parents)
     if (p / 'scripts/mission3_baseline.py').is_file()),
    None,
)
if REPO_ROOT is None and Path('/content/prelim/scripts/mission3_baseline.py').is_file():
    REPO_ROOT = Path('/content/prelim')
if REPO_ROOT is None:
    raise FileNotFoundError('prelim 저장소를 준비하고 그 안에서 노트북을 실행하세요.')

PYTHON_EXECUTABLE = os.environ.get('DCC_PYTHON', sys.executable)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)
print('Python:', PYTHON_EXECUTABLE)
# requirements.txt 설치는 선택한 Python 환경에서 먼저 완료해 주세요.


## 데이터 경로 설정

개인 Google Drive 경로를 노트북에 고정하지 않습니다. `DCC_DATA_DIR`에는 주최 측이 제공한 대학부 데이터의 **로컬 루트 폴더**를 지정합니다.

`DCC_DATA_ROOT`(기존 로컬 설정) 또는 `DCC_DATA_DIR`에 데이터 루트를 지정하세요. 둘 다 있으면 `DCC_DATA_ROOT`를 우선합니다. 기본값은 Colab의 `/content/data`, 로컬의 `data/raw_original`입니다. 이 노트북은 저장소를 자동 다운로드하거나 Drive를 자동 마운트하지 않습니다.


In [ ]:
from pathlib import Path
import os

# Colab/로컬 환경에서 공식 데이터가 있는 루트만 지정합니다.
# 특정 팀원의 개인 저장경로를 노트북에 하드코딩하지 않습니다.
default_data = '/content/data' if Path('/content').is_dir() else REPO_ROOT / 'data/raw_original'
DATA_DIR = Path(os.environ.get('DCC_DATA_ROOT', os.environ.get('DCC_DATA_DIR', default_data))).expanduser().resolve()
ARTIFACT_DIR = REPO_ROOT / 'artifacts'
PREDICTION_PATH = REPO_ROOT / 'outputs/mission3.csv'

TRAIN_LABEL_DIR = DATA_DIR / 'Training' / '2.라벨링데이터' / 'TL_서울_구급'
VALID_LABEL_DIR = DATA_DIR / 'Validation' / '2.라벨링데이터' / 'VL_서울_구급'
VALID_AUDIO_DIR = DATA_DIR / 'Validation' / '1.원천데이터' / 'VS_서울_구급'

print('DATA_DIR:', DATA_DIR)
print('Training JSON 폴더:', TRAIN_LABEL_DIR, TRAIN_LABEL_DIR.exists())
print('Validation JSON 폴더:', VALID_LABEL_DIR, VALID_LABEL_DIR.exists())
print('Validation WAV 폴더:', VALID_AUDIO_DIR, VALID_AUDIO_DIR.exists())

assert TRAIN_LABEL_DIR.exists(), 'Training JSON 폴더를 찾지 못했습니다. DCC_DATA_DIR를 확인하세요.'


In [ ]:
# 하위 폴더까지 세어 실제 입력 개수를 확인합니다.
train_json_files = sorted(TRAIN_LABEL_DIR.rglob('*.json'))
valid_json_files = sorted(VALID_LABEL_DIR.rglob('*.json'))
print(f'Training JSON: {len(train_json_files):,}개')
print(f'Validation JSON: {len(valid_json_files):,}개')
assert train_json_files, 'Training JSON을 찾지 못했습니다.'
assert valid_json_files, 'Validation JSON을 찾지 못했습니다.'


In [ ]:
# 샘플의 증상과 전사 텍스트만 확인합니다. 빈 발화 목록도 처리합니다.
import json

with train_json_files[0].open('r', encoding='utf-8-sig') as file:
    sample_data = json.load(file)
utterances = sample_data.get('utterances', [])
utterances = utterances if isinstance(utterances, list) else []
first_text = next((item.get('text', '') for item in utterances if isinstance(item, dict)), '')
print('symptom:', sample_data.get('symptom'))
print('발화 개수:', len(utterances))
print('첫 번째 발화 text:', first_text)


In [ ]:
# 현재 main의 baseline 함수와 동일한 이름을 사용합니다.
from scripts.mission3_baseline import load_training_dataframe, make_dataset_summary

dataframe = load_training_dataframe(TRAIN_LABEL_DIR)
summary = make_dataset_summary(dataframe)

print('불러온 데이터 개수:', len(dataframe))
print('빈 전사 수:', summary['empty_text_count'])
print('목표 9개 증상이 하나도 없는 샘플 수:', summary['zero_target_label_count'])
display(dataframe.head())
print('JSON 읽기 실패 수:', summary['load_failure_count'])
assert summary['load_failure_count'] == 0, '읽기 실패한 Training JSON을 먼저 확인하세요.'


In [ ]:
# 학습에 실제로 쓰이는 9개 증상과 그 외 symptom을 확인합니다.
display(
    dataframe[['label_file_name', 'symptoms', 'non_target_symptoms']].head(10)
)


In [ ]:
# 실패하면 즉시 오류를 표시하여 이전 실행의 결과를 읽지 않게 합니다.
subprocess.run([
    PYTHON_EXECUTABLE, str(REPO_ROOT / 'scripts/mission3_baseline.py'),
    '--label_dir', str(TRAIN_LABEL_DIR), '--output_dir', str(ARTIFACT_DIR),
    '--threshold', '0.5', '--test_size', '0.2', '--seed', '42',
], cwd=REPO_ROOT, check=True)


In [ ]:
# 학습 결과가 공식 threshold 규칙을 지켰는지 확인합니다.
import json

with open(ARTIFACT_DIR / 'metrics.json', 'r', encoding='utf-8') as file:
    metrics = json.load(file)

print('Validation Macro F1:', metrics['macro_f1'])
print('Decision Threshold:', metrics['threshold'])
assert metrics['threshold'] == 0.5, '공식 규정상 threshold는 반드시 0.5여야 합니다.'


In [ ]:
# 공식 Validation 정답은 사용하지 않고 전사 텍스트로 추론합니다.
assert VALID_LABEL_DIR.is_dir(), 'Validation JSON 폴더를 찾지 못했습니다.'
assert VALID_AUDIO_DIR.is_dir(), 'Validation WAV 폴더를 찾지 못했습니다.'
subprocess.run([
    PYTHON_EXECUTABLE, str(REPO_ROOT / 'scripts/inference.py'),
    '--audio_dir', str(VALID_AUDIO_DIR), '--label_dir', str(VALID_LABEL_DIR),
    '--ckpt_path', str(ARTIFACT_DIR / 'mission3_tfidf_logreg.joblib'),
    '--output', str(PREDICTION_PATH),
], cwd=REPO_ROOT, check=True)


In [ ]:
# 생성된 Mission 3 결과 형식을 확인합니다.
import pandas as pd

submission = pd.read_csv(PREDICTION_PATH)
validation_json_count = len(list(VALID_LABEL_DIR.rglob('*.json')))

print('예측 행 수:', len(submission))
print('Validation JSON 수:', validation_json_count)
display(submission.head())

assert list(submission.columns) == ['label file name', 'symptom'], 'Mission 3 CSV 열 이름을 확인하세요.'
assert len(submission) == validation_json_count, '입력 JSON 수와 출력 CSV 행 수가 다릅니다.'


## 제출 전 주의

이 PR의 자동 검사는 코드 구조와 기본 테스트를 확인하는 것이며, 실제 대회 데이터 실행을 대신하지 않습니다. 최종 제출용 학습 노트북은 **공식 Training 데이터로 실제 실행한 학습 로그와 주요 결과가 남아 있는 상태**로 저장하고, `inference.py` 단일 실행으로 결과 CSV가 생성되는지 다시 확인합니다.
